Preprocessing AnnData ZARR

AnnData ZARR Package

In [3]:
!pip install anndata zarr numpy open3d


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\ryash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [16]:
!pip install matplotlib scipy

     ---------------------------------------- 0.0/52.8 kB ? eta -:--:--
     ------------------------------------ - 51.2/52.8 kB 871.5 kB/s eta 0:00:01
     ------------------------------------ - 51.2/52.8 kB 871.5 kB/s eta 0:00:01
     -------------------------------------- 52.8/52.8 kB 340.8 kB/s eta 0:00:00
     ---------------------------------------- 0.0/119.8 kB ? eta -:--:--
     -------------------------------------- 119.8/119.8 kB 3.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/8.2 MB ? eta -:--:--
   ---- ----------------------------------- 1.0/8.2 MB 30.4 MB/s eta 0:00:01
   ---------------- ----------------------- 3.3/8.2 MB 42.8 MB/s eta 0:00:01
   -------------------------- ------------- 5.4/8.2 MB 43.3 MB/s eta 0:00:01
   ------------------------------------- -- 7.7/8.2 MB 44.9 MB/s eta 0:00:01
   ---------------------------------------  8.2/8.2 MB 43.8 MB/s eta 0:00:01
   ---------------------------------------- 8.2/8.2 MB 35.0 MB/s eta 0:00:00
   -


[notice] A new release of pip is available: 24.0 -> 26.1
[notice] To update, run: C:\Users\ryash\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [17]:
import anndata as ad
import pandas as pd
import numpy as np

# Load
adata = ad.read_zarr("reg001_expr-anndata.zarr")

coords = adata.obsm["xy"]
labels = adata.obs["Nuclei K-Means [Mean] Expression"]

# Convert labels cleanly
labels = labels.astype(int).astype(str)

# OPTIONAL: downsample (CDE can lag/fail with ~50k points)
# keep every 2nd point (~25k)
coords = coords[::2]
labels = labels.iloc[::2]

# Build DataFrame
df = pd.DataFrame({
    "X": coords[:, 0],
    "Y": coords[:, 1],
    "Cell Type": labels.values
})

# Save
output_path = "cells_cde_ready.csv"
df.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("Rows:", len(df))
print("Unique labels:", sorted(set(df["Cell Type"])))
print(df.head())

Saved: cells_cde_ready.csv
Rows: 24747
Unique labels: ['1', '3', '4', '5', '6', '8', '9']
             X          Y Cell Type
0  6296.351703  13.374104         3
1  6182.411859  15.535256         3
2  7193.487253  16.786195         3
3  3258.838608  17.611142         3
4  6689.803783  18.153073         3


In [2]:
print(adata.obs.columns)

Index(['Cell K-Means [Mean] Expression',
       'Cell K-Means [Covariance] Expression',
       'Cell K-Means [Total] Expression',
       'Cell K-Means [Mean-All-SubRegions] Expression',
       'Cell K-Means [Shape-Vectors]', 'Cell K-Means [Texture]',
       'Cell K-Means [tSNE_All_Features]',
       'Cell K-Means [Shape-Vectors Normalized]',
       'Cell K-Means [UMAP_All_Features]', 'Nuclei K-Means [Mean] Expression',
       'Nuclei K-Means [Covariance] Expression',
       'Nuclei K-Means [Total] Expression',
       'Nuclei K-Means [Mean-All-SubRegions] Expression',
       'Nuclei K-Means [Shape-Vectors]', 'Nuclei K-Means [Texture]',
       'Nuclei K-Means [tSNE_All_Features]',
       'Nuclei K-Means [Shape-Vectors Normalized]',
       'Nuclei K-Means [UMAP_All_Features]',
       'Cell Boundaries K-Means [Mean] Expression',
       'Cell Boundaries K-Means [Covariance] Expression',
       'Cell Boundaries K-Means [Total] Expression',
       'Cell Boundaries K-Means [Mean-All-SubRegions

In [3]:
for col in adata.obs.columns:
    print(col, adata.obs[col].nunique())

Cell K-Means [Mean] Expression 3
Cell K-Means [Covariance] Expression 3
Cell K-Means [Total] Expression 3
Cell K-Means [Mean-All-SubRegions] Expression 8
Cell K-Means [Shape-Vectors] 3
Cell K-Means [Texture] 1
Cell K-Means [tSNE_All_Features] 3
Cell K-Means [Shape-Vectors Normalized] 4
Cell K-Means [UMAP_All_Features] 3
Nuclei K-Means [Mean] Expression 9
Nuclei K-Means [Covariance] Expression 3
Nuclei K-Means [Total] Expression 3
Nuclei K-Means [Mean-All-SubRegions] Expression 8
Nuclei K-Means [Shape-Vectors] 3
Nuclei K-Means [Texture] 1
Nuclei K-Means [tSNE_All_Features] 3
Nuclei K-Means [Shape-Vectors Normalized] 4
Nuclei K-Means [UMAP_All_Features] 3
Cell Boundaries K-Means [Mean] Expression 3
Cell Boundaries K-Means [Covariance] Expression 4
Cell Boundaries K-Means [Total] Expression 3
Cell Boundaries K-Means [Mean-All-SubRegions] Expression 8
Cell Boundaries K-Means [Shape-Vectors] 3
Cell Boundaries K-Means [Texture] 1
Cell Boundaries K-Means [tSNE_All_Features] 3
Cell Boundaries 

In [16]:
print(labels.value_counts())

Nuclei K-Means [Mean] Expression
3    15915
1     5115
4     2500
6      915
8      277
5       24
9        1
Name: count, dtype: int64


In [15]:
print("Coordinate range:")
print("X:", coords[:,0].min(), "→", coords[:,0].max())
print("Y:", coords[:,1].min(), "→", coords[:,1].max())

Coordinate range:
X: 16.4656007751938 → 9972.195205479453
Y: 13.374103942652331 → 9486.787455197133


In [2]:
import pandas as pd

# Load input CSV
df = pd.read_csv("cells_xy_preprocessed.csv")

# Ensure required columns exist (adjust names if needed)
# Assumes input already has x, y, and Cell Type
df = df.rename(columns={
    "X": "x",
    "Y": "y",
    "cell_type": "Cell Type"
})

# Create output with exact columns
output = pd.DataFrame({
    "x": df["x"],
    "y": df["y"],
    "z": 0,  # fill all z with 0
    "Cell Type": df["Cell Type"]
})

# Save result
output.to_csv("cells_xyz_formatted.csv", index=False)

print("Saved: cells_xyz_formatted.csv")
print(output.head())

Saved: cells_xyz_formatted.csv
             x          y  z  Cell Type
0  6296.351703  13.374104  0          3
1  6182.411859  15.535256  0          3
2  7193.487253  16.786195  0          3
3  3258.838608  17.611142  0          3
4  6689.803783  18.153073  0          3
